# Imports

In [1]:
# patch the old packages calling visa
# this is a hack to make the old packages work with pyvisa

#############################################################################################################################
import pyvisa

# Simulate the `visa` module as an alias for `pyvisa`
import sys

# Create a fake 'visa' module, which is essentially an alias for pyvisa
sys.modules['visa'] = pyvisa

# Optionally, map all attributes from pyvisa to visa (this is technically unnecessary because the alias works)
for attr in dir(pyvisa):
    setattr(sys.modules['visa'], attr, getattr(pyvisa, attr))

#############################################################################################################################


In [5]:
# Make the sagnac warning flags show up in the notebook


import logging

# Configure logging to output to the notebook's cell
logging.basicConfig(level=logging.WARNING,  # Log only WARNING or above by default
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    handlers=[logging.StreamHandler()])


In [2]:
# MAGNET CONTROL
from custom_instruments import daedalusProjField
from pymeasure.adapters import DAQmxAdapter

calib_file = 'C:\\Users\\Ralph Group\\Documents\\Github\\SagnacOperatingSys\\sagnac_control\\calibrations\\sagnac'

magnet = daedalusProjField(DAQmxAdapter('Dev1', ['ao0', 'ai1']),"GPIB::10")
magnet.load_calibration_params(calib_file)

# magnet.set_vector_field(
#     B=0,
#     phi=0, 
#     theta=0)

In [131]:
# LCR METER
from pymeasure.instruments.agilent import Agilent4284A
LCR = Agilent4284A("GPIB::16")
LCR.reset()
LCR.adapter.connection.timeout = 30000
LCR.frequency


1000.0

In [4]:
# GENERICS
import os
import time

import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import plotly.express as px
from IPython.display import display, clear_output



# Basic Comands

In [66]:
LCR.impedance_mode = "RX"
a,b,f = LCR.sweep_measurement(
    'frequency', decadeFreqs)     
# no wait needed for the LCR meter, as it blocks until the measurement is done

2.0794589519500732


In [75]:
magnet.set_vector_field(B=0.15,phi=-160,theta=0)
while magnet.in_motion:
    pass

In [78]:
# print magnet field
print(  magnet.phi )
print( magnet.theta )
print( magnet.field )

-160.0
0.0
0.0


# Playground

In [12]:
calFreqs = [20, 25, 30, 40, 50, 60, 80, 100, 120, 150, 200, 250, 300, 400, 500, 600, 800,
 1000, 1200, 1500, 2000, 2500, 3000, 4000, 5000, 6000, 8000, 10000, 12000, 15000, 
 20000, 25000, 30000, 40000, 50000, 60000, 80000, 100000, 120000, 150000, 200000, 
 250000, 300000, 400000, 500000, 600000, 800000, 1000000]
 
decadeFreqs = np.logspace(2, 6, 5); print(decadeFreqs)

[1.e+02 1.e+03 1.e+04 1.e+05 1.e+06]


In [136]:
np.arange(15,1,-1)

array([15, 14, 13, 12, 11, 10,  9,  8,  7,  6,  5,  4,  3,  2])

In [141]:
file_path = r"C:\Users\Ralph Group\Documents\Data\Orion\calibration\LCRamrTest\test.h5"
experiment  = "postCalibration/281ohm/PtCO/devC1R2/compSOL/noCustomFreqs/Bz_sweep"

freqs = calFreqs
v_applied = 1.0
theta = 90
# B = 0.15

hystersis = lambda x: np.concatenate((x, -x))  
for phi in  [magnet.phi]: # hystersis( np.arange(-170, 171, 10) ):   # hystersis( np.arange(-170, 171, 10) ):  # in [magnet.phi]:
    for B in hystersis( np.arange(-0.17,0.18, 0.01) ): #np.arange(0.03, 0.2, 0.03):

        # Set the magnetic field
        magnet.set_vector_field(B=B,phi=phi,theta=theta)
        while magnet.in_motion:
            pass

        # Record LCR data
        # LCR.ac_voltage = v_applied
        LCR.impedance_mode = "RX"
        LCR.ac_voltage = 1
        a,b,f = LCR.sweep_measurement(
            'frequency', freqs )

        # Create a DataFrame with frequency (f) and complex data
        df = pd.DataFrame({'phi':phi,'f': f, 'a': a, 'b': b})
        df["phasor"] = df.a + 1j*df.b
        df["time"]= time.time()    
        df["B"] = B 
        df["v_applied"] = v_applied

        #Save Data
        # df.to_csv(file_path, mode='a', header=not os.path.isfile(file_path), index=False)
        df.to_hdf(file_path, key=experiment,                  append=True, format='table')

        # # Plot Data
        # dfplt = pd.read_hdf(file_path, key=experiment)
        # fig = px.scatter(dfplt, x='f', y='a', color='phi',log_x=True)
        # clear_output(wait=True)
        # display(fig)

        clear_output(wait=True)


        # Plot first figure (a vs f)
        dfplt = pd.read_hdf(file_path, key=experiment)
        fig1 = px.scatter(dfplt, x='f', y='a', color='B', log_x=True)
        display(fig1)

        # Plot second figure (b vs f)
        dfplt = pd.read_hdf(file_path, key=experiment)
        fig2 = px.scatter(dfplt, x='f', y='b', color='B', log_x=True)
        display(fig2)

In [45]:
magnet.set_vector_field(B=0,phi=0,theta=0)

# File management

In [142]:
if input(f"add metadata to {experiment}? (y/n)") == 'y':
    with pd.HDFStore(file_path) as store:
        storer = store.get_storer(experiment)
        storer.attrs.readme = getattr(storer.attrs, 'readme', '') + "\n" +input("Add to README: ")

        # storer.attrs.metadata = input(f"replace {getattr(storer.attrs, 'metadata', '')} metadata: (dictionary format) ")
        #{
        #     'author': 'John Doe',
        #     'date': '2018-01-01',
        #     'description': 'Random dataset'
        # }


with pd.HDFStore(file_path) as store:
    for key in store.keys():
        storer = store.get_storer(key)

        print(f"\n____________________________\nKey: {key}")
        print("readme: \n" + getattr(storer.attrs, 'readme', "No README available"))
        # print("\nmetadata: \n" + getattr(storer.attrs, 'metadata', "No metadata available"))


____________________________
Key: /LowRes_sweepB
readme: 
junk

____________________________
Key: /LowRes_sweepB2
readme: 

oops. B field swept but not recorded

____________________________
Key: /LowRes_sweepB3
readme: 
No README available

____________________________
Key: /NoHall
readme: 
No README available

____________________________
Key: /NoHall_revSweep
readme: 
No README available

____________________________
Key: /postCalibration/100ohm
readme: 

short Open Load compensation applied

____________________________
Key: /postCalibration/330ohm5/PtCO/devRand/compSOL/noCustomFreqs
readme: 

Column 10, row 8, the device is 20x80 - 4x4 square

____________________________
Key: /postCalibration/281ohm/PtCO/devC1R2/compSOL/noCustomFreqs
readme: 
No README available

____________________________
Key: /postCalibration/281ohm/PtCO/devC1R2/compSOL/noCustomFreqs/Bz_sweep
readme: 
No README available

____________________________
Key: /postCalibration/281ohm/PtCO/devC1R2/compSOL/noCustom